# Model Explainability - TreeSHAP for the Final XGBoost Model

The final XGBoost model using the Day-Anchored Lookback Strategy (trained on train+val, tuned lookback = 24h)
is the best performing model. This notebook makes it interpretable using TreeSHAP
: exact Shapley values for tree ensembles, providing local
explanations per hour that aggregate consistently into global feature importance.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import holidays
import shap
from sklearn.preprocessing import RobustScaler
from xgboost import XGBRegressor

## Load Data & Rebuild the Pipeline (identical to `modelling.ipynb`)

In [ ]:
# ---------- Load data ----------
data = pd.read_parquet("data/ENTSOE/entsoe_price_and_loadfc.parquet")
weather_fc = pd.read_parquet("data/NOAA GFS/weather_forecast.parquet")
data = data.join(weather_fc, how="left")

# ---------- Feature extraction (identical to modelling.ipynb) ----------
# lag features (naive timestamp manipulation to avoid issues with DST transitions)
idx_naive = data.index.tz_localize(None)
prev_naive_day = idx_naive - pd.DateOffset(days=1)
prev_naive_day2 = idx_naive - pd.DateOffset(days=2)
prev_naive_week = idx_naive - pd.DateOffset(weeks=1)

idx_prev_day = prev_naive_day.tz_localize("Europe/Berlin", ambiguous=False, nonexistent="shift_forward")
idx_prev_day2 = prev_naive_day2.tz_localize("Europe/Berlin", ambiguous=False, nonexistent="shift_forward")
idx_prev_week = prev_naive_week.tz_localize("Europe/Berlin", ambiguous=False, nonexistent="shift_forward")

data["price_lag_24h"] = data["da_price"].reindex(idx_prev_day).to_numpy()
data["price_lag_48h"] = data["da_price"].reindex(idx_prev_day2).to_numpy()
data["price_lag_168h"] = data["da_price"].reindex(idx_prev_week).to_numpy()

# calendar features in local delivery time
data["weekday_sin"] = np.sin(2 * np.pi * data.index.dayofweek / 7)
data["weekday_cos"] = np.cos(2 * np.pi * data.index.dayofweek / 7)
data["hour_sin"] = np.sin(2 * np.pi * data.index.hour / 24)
data["hour_cos"] = np.cos(2 * np.pi * data.index.hour / 24)

ger_holidays = list(holidays.Germany(years=data.index.year.unique()))
ger_holidays = pd.to_datetime(ger_holidays).tz_localize(data.index.tz)
data["holiday_not_sunday"] = (data.index.normalize().isin(ger_holidays) & (data.index.dayofweek != 6))

data = data["2023":"2025"]

# ---------- Constants (identical to modelling.ipynb) ----------
LOCAL_TZ = "Europe/Berlin"
TRAIN_START = pd.Timestamp("2023-01-01", tz=LOCAL_TZ)
VAL_END     = pd.Timestamp("2025-07-01", tz=LOCAL_TZ)   # train+val = TRAIN_START .. VAL_END
TEST_START  = pd.Timestamp("2025-07-01", tz=LOCAL_TZ)
TEST_END    = pd.Timestamp("2026-01-01", tz=LOCAL_TZ)

TARGET_COL = "da_price"
FEATURE_COLS = [
    "load_fc", "ssrd_fc", "U100_fc",
    "price_lag_24h", "price_lag_48h", "price_lag_168h",
    "weekday_sin", "weekday_cos", "hour_sin", "hour_cos", "holiday_not_sunday",
]  # final feature set after feature selection in modelling.ipynb
LAG_COLS = [col for col in FEATURE_COLS if col.startswith("price_lag")]
SCALE_COLS = ["load_fc", "price_lag_24h", "price_lag_48h", "price_lag_168h", "ssrd_fc", "U100_fc"]
SEQUENCE_COLS = [TARGET_COL]
XGB_LOOKBACK = 24   # tuned lookback of the final model

model_df = data[[TARGET_COL] + FEATURE_COLS].copy()

# helper functions for transformation and error calculation
def asinh_transform(y, c):
    return np.arcsinh(y / c)

def asinh_inverse(y_t, c):
    return c * np.sinh(y_t)

def mae_score(y_true, y_pred) -> float:
    return float(np.mean(np.abs(np.asarray(y_true) - np.asarray(y_pred))))

def rmse_score(y_true, y_pred) -> float:
    return float(np.sqrt(np.mean((np.asarray(y_true) - np.asarray(y_pred)) ** 2)))

In [ ]:
# ---------- Stage-2 preprocessing: c and scaler fitted on RAW train+val ----------
trainval_df = model_df.loc[(model_df.index >= TRAIN_START) & (model_df.index < VAL_END)].copy()
c_tv = np.median(np.abs(trainval_df[TARGET_COL]))
for col in [TARGET_COL] + LAG_COLS:
    trainval_df[col] = asinh_transform(trainval_df[col], c_tv)
scaler_tv = RobustScaler()
trainval_df[SCALE_COLS] = scaler_tv.fit_transform(trainval_df[SCALE_COLS])
print(f"c_tv = {c_tv:.4f}")

# ---------- Test window (history buffer + test period), stage-2 preprocessing ----------
test_win = model_df.loc[
    (model_df.index >= TEST_START - pd.Timedelta(hours=XGB_LOOKBACK)) & (model_df.index < TEST_END)
].copy()
y_test = test_win.loc[test_win.index >= TEST_START, TARGET_COL].to_numpy(dtype=np.float32)  # raw EUR/MWh
for col in [TARGET_COL] + LAG_COLS:
    test_win[col] = asinh_transform(test_win[col], c_tv)
test_win[SCALE_COLS] = scaler_tv.transform(test_win[SCALE_COLS])

# ---------- Day-Anchored Lookback Strategy builder (identical to modelling.ipynb) ----------
def _date_ints_from_index(index: pd.DatetimeIndex) -> np.ndarray:
    return np.array(
        [ts.year * 10000 + ts.month * 100 + ts.day for ts in index],
        dtype=np.int32,
    )

def build_anchored_dataset(df, sequence_cols, exog_cols, target_col, lookback_hours):
    """
    For each target timestamp in local delivery day D, the lookback window covers
    the rows ending immediately before D starts (same day-anchored history window for all
    hours of the same delivery day, DST-safe).
    """
    idx = df.index
    sequence_values = df[sequence_cols].to_numpy(dtype=np.float32)
    exog_values = df[exog_cols].to_numpy(dtype=np.float32)
    target_values = df[target_col].to_numpy(dtype=np.float32)
    date_ints = _date_ints_from_index(idx)

    first_pos_by_date = {}
    for pos, date_int in enumerate(date_ints):
        first_pos_by_date.setdefault(date_int, pos)

    X_seq, X_exog, y_target, ts_target = [], [], [], []
    for row_idx in range(len(df)):
        day_start_pos = first_pos_by_date[date_ints[row_idx]]
        keep_idx = np.arange(day_start_pos - lookback_hours, day_start_pos)
        if keep_idx[0] < 0:
            continue
        X_seq.append(sequence_values[keep_idx, :])
        X_exog.append(exog_values[row_idx])
        y_target.append(target_values[row_idx])
        ts_target.append(idx[row_idx])

    return (
        np.stack(X_seq),
        np.stack(X_exog),
        np.asarray(y_target, dtype=np.float32),
        pd.DatetimeIndex(ts_target),
    )

seq, exog, _, ts = build_anchored_dataset(test_win, SEQUENCE_COLS, FEATURE_COLS, TARGET_COL, XGB_LOOKBACK)
keep = ts >= TEST_START
seq, exog, ts_test = seq[keep], exog[keep], ts[keep]

# flat input as a DataFrame with readable feature names:
# window position "price_h-k" = asinh price k hours before the start of the delivery day
window_names = [f"price_h-{XGB_LOOKBACK - i}" for i in range(XGB_LOOKBACK)]
feature_names = window_names + FEATURE_COLS
X_test = pd.DataFrame(
    np.concatenate([seq.reshape(seq.shape[0], -1), exog], axis=1),
    columns=feature_names,
    index=ts_test,
)
print("X_test:", X_test.shape, "\ny_test:", y_test.shape)

In [ ]:
# ---------- Load the final model + sanity checks ----------
xgb_anchored_final = XGBRegressor()
xgb_anchored_final.load_model("models/xgb_anchored_final.ubj")

assert xgb_anchored_final.n_features_in_ == X_test.shape[1], (
    f"feature mismatch: model expects {xgb_anchored_final.n_features_in_}, data has {X_test.shape[1]}"
)

y_pred = asinh_inverse(xgb_anchored_final.predict(X_test.to_numpy(dtype=np.float32)), c_tv)
print(f"Reproduced test scores: MAE={mae_score(y_test, y_pred):.4f}  RMSE={rmse_score(y_test, y_pred):.4f}")

## TreeSHAP

In [ ]:
explainer = shap.TreeExplainer(xgb_anchored_final)   # tree_path_dependent (exact, no background data)
sv = explainer(X_test)                      # Explanation object (values, base_values, data)

# additivity check: base + sum(SHAP) must equal the model output (asinh scale)
pred_t = xgb_anchored_final.predict(X_test.to_numpy(dtype=np.float32))
recon = np.ravel(sv.base_values) + sv.values.sum(axis=1)
print("max |base + sum(SHAP) - prediction| =", float(np.abs(recon - pred_t).max()))
print(f"base value (expected asinh price) = {float(np.ravel(sv.base_values)[0]):.4f}")

In [ ]:
# ---------- Collapse cyclic pairs (hour_sin/cos, weekday_sin/cos) into one feature each ----------
# SHAP is additive, so the combined contribution of a cyclic pair is the SUM of its two
# SHAP values (per instance). Additivity is preserved: base + sum stays the prediction
# (asinh scale). The data column for the merged feature is the ACTUAL value
# (hour 0-23, weekday 0-6) so beeswarm/dependence colour and axis are meaningful.
cyclic_pairs = {
    "hour":    ("hour_sin", "hour_cos", np.asarray(ts_test.hour, dtype=float)),
    "weekday": ("weekday_sin", "weekday_cos", np.asarray(ts_test.dayofweek, dtype=float)),
}
drop = {c for a, b, _ in cyclic_pairs.values() for c in (a, b)}

keep_cols = [f for f in feature_names if f not in drop]
keep_idx = [feature_names.index(f) for f in keep_cols]
collapsed_names = keep_cols + list(cyclic_pairs)

vals = [sv.values[:, keep_idx]]
dat  = [X_test.iloc[:, keep_idx].to_numpy()]
for name, (a, b, actual) in cyclic_pairs.items():
    ia, ib = feature_names.index(a), feature_names.index(b)
    vals.append((sv.values[:, ia] + sv.values[:, ib])[:, None])   # summed SHAP (net contribution)
    dat.append(actual[:, None])                                   # actual hour / weekday
collapsed_vals = np.concatenate(vals, axis=1)
collapsed_data = np.concatenate(dat, axis=1)
abs_c = np.abs(collapsed_vals)

sv_c = shap.Explanation(
    values=collapsed_vals,
    base_values=sv.base_values,
    data=collapsed_data,
    feature_names=collapsed_names,
)

# additivity preserved (only regrouped): base + sum(collapsed) == prediction (asinh scale)
recon_c = np.ravel(sv_c.base_values) + collapsed_vals.sum(axis=1)
print("max |base + sum(collapsed SHAP) - prediction| =", float(np.abs(recon_c - pred_t).max()))
print(f"collapsed features: {len(collapsed_names)} (was {len(feature_names)})")


### Global feature importance

First the aggregated view (the 24 price-history window hours summed into one bar, so it is comparable to the named features), then the per-window-hour breakdown (recency structure), then the beeswarm for the direction of effects.

In [ ]:
# ---------- Global importance ----------
# mean |SHAP| per feature (asinh scale). The 24 price-history window hours are summed
# into ONE bar (total contribution) so it is comparable to the individual named features.
win_idx  = [collapsed_names.index(w) for w in window_names]
exo_names = [f for f in collapsed_names if f not in window_names]

agg = {f"price history ({XGB_LOOKBACK}h)": np.abs(sv_c.values[:, win_idx]).sum(axis=1).mean()}
for f in exo_names:
    agg[f] = np.abs(sv_c.values[:, collapsed_names.index(f)]).mean()
agg = pd.Series(agg).sort_values()

fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(agg.index, agg.values, color="tab:blue")
for i, v in enumerate(agg.values):
    ax.text(v, i, f"  {v:.4f}", va="center")
ax.set_xlabel("mean |SHAP| (asinh scale)")
ax.set_title("Global feature importance (price history aggregated)")
plt.tight_layout()
plt.show()

# ---------- Beeswarm: distribution and direction of the effects ----------
shap.plots.beeswarm(sv_c, max_display=15)

# ---------- Price-history breakdown: importance per window hour (recency structure) ----------
win_imp = np.abs(sv_c.values[:, win_idx]).mean(axis=0)      # mean |SHAP| per window position
lags = [XGB_LOOKBACK - i for i in range(XGB_LOOKBACK)]      # hours before the delivery day (24 .. 1)

fig, ax = plt.subplots(figsize=(11, 4))
ax.bar(lags, win_imp, color="tab:orange")
ax.invert_xaxis()                                          # most recent (h-1) on the right
ax.set_xlabel("hours before the delivery day (window position)")
ax.set_ylabel("mean |SHAP| (asinh scale)")
ax.set_title("Price history breakdown: importance per window hour")
plt.tight_layout()
plt.show()


Global importance (price history aggregated)
- Wind (`U100`) is the strongest single driver: one feature almost matches the entire 24 h price window, which is a sum over 24 features.
- The three explicit same-hour lags together (0.1375) and the full window (0.1369) have almost identical global importance.
- The weekly lag (168 h; 0.0472) is more important than the 2-day lag (48 h; 0.0281), consistent with the weekly peak seen in the ACF during the EDA.
- `holiday_not_sunday` has a very low value, but mean |SHAP| averages over all test hours and therefore dilutes rare events (~10 days per year). Its model attribution is therefore better examined with local explanations.

Price history breakdown
- The window contribution is concentrated in the most recent observations: the last hour before the delivery day (h-1) accounts for about 21 % of it, and h-1 and h-2 together account for about 32 %.
- A secondary bump appears 8-10 hours before the delivery day (~14:00-16:00 on the previous day), coinciding with typically solar-active hours.

Beeswarm
- High wind and high solar radiation are associated with negative SHAP values, consistent with the merit-order effect. Both relationships are asymmetric.
- High load forecasts and high previous prices are associated with positive SHAP values.
- Temporal features generally have lower SHAP magnitudes than the other feature categories.

In [ ]:
# ---------- Grouped importance (on the collapsed features) ----------
# hour/weekday are already the NET (|sum|) contribution of their sin/cos pair.
groups = {
    f"price history window ({XGB_LOOKBACK}h)": window_names,
    "price lags (24/48/168h)": LAG_COLS,
    "load forecast": ["load_fc"],
    "weather forecasts (ssrd_fc, U100_fc)": ["ssrd_fc", "U100_fc"],
    "calendar": ["hour", "weekday", "holiday_not_sunday"],
}

rows = []
for name, cols in groups.items():
    idx = [collapsed_names.index(c) for c in cols]
    rows.append({"group": name, "mean_abs_shap": abs_c[:, idx].sum(axis=1).mean()})
group_df = pd.DataFrame(rows).sort_values("mean_abs_shap", ascending=True, ignore_index=True)
group_df["share_pct"] = 100 * group_df["mean_abs_shap"] / group_df["mean_abs_shap"].sum()

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.barh(group_df["group"], group_df["mean_abs_shap"])
for i, (v, p) in enumerate(zip(group_df["mean_abs_shap"], group_df["share_pct"])):
    ax.text(v, i, f"  {p:.1f}%", va="center")
ax.set_xlabel("mean |SHAP| (asinh scale)")
ax.set_title("Grouped feature importance")
plt.tight_layout()
plt.show()

display(group_df.sort_values("mean_abs_shap", ascending=False, ignore_index=True))

In [ ]:
# ---------- Window position x target hour ----------
# The window ends immediately before the delivery day starts.
win_idx_c = [collapsed_names.index(w) for w in window_names]
target_hour = np.asarray(ts_test.hour)
lag24_idx = collapsed_names.index("price_lag_24h")

heat = np.zeros((24, XGB_LOOKBACK))
heat_comb = np.zeros((24, XGB_LOOKBACK))
same_hour_positions = []

for h in range(24):
    mask = target_hour == h
    if not mask.any():
        continue
    win_vals = sv_c.values[np.ix_(mask, win_idx_c)]
    heat[h] = np.abs(win_vals).mean(axis=0)
    combined = win_vals.copy()

    # D-1 at target hour h is in the window only if this position is available.
    same_hour_pos = XGB_LOOKBACK - 24 + h
    if 0 <= same_hour_pos < XGB_LOOKBACK:
        combined[:, same_hour_pos] += sv_c.values[mask, lag24_idx]
        same_hour_positions.append((same_hour_pos, h))
    heat_comb[h] = np.abs(combined).mean(axis=0)

vmax = max(heat.max(), heat_comb.max())
fig, axes = plt.subplots(1, 2, figsize=(16, 7), sharey=True)
for ax, matrix, title in [
    (axes[0], heat, "window only (credit split with price_lag_24h)"),
    (axes[1], heat_comb, "window + price_lag_24h merged where available"),
]:
    im = ax.imshow(
        matrix, aspect="auto", cmap="viridis", origin="lower", vmin=0, vmax=vmax
    )
    if same_hour_positions:
        positions, hours = zip(*same_hour_positions)
        ax.plot(
            positions,
            hours,
            "w--",
            lw=1,
            alpha=0.7,
            label="same clock hour on D-1",
        )
        ax.legend(loc="upper left")
    tick_pos = np.unique(
        np.linspace(0, XGB_LOOKBACK - 1, min(XGB_LOOKBACK, 9), dtype=int)
    )
    ax.set_xticks(tick_pos)
    ax.set_xticklabels([f"h-{XGB_LOOKBACK - pos}" for pos in tick_pos])
    ax.set_xlabel("window position relative to delivery-day start")
    ax.set_title(title)

axes[0].set_ylabel("target hour of the delivery day")
fig.colorbar(
    im,
    ax=axes,
    label="mean |SHAP| (asinh scale)",
    fraction=0.046,
    pad=0.04,
)
fig.suptitle("Which history-window position explains which delivery hour?")
plt.show()

lag_rows = [
    {
        "information": f"price-history window ({XGB_LOOKBACK} values)",
        "mean_abs_shap": np.abs(sv_c.values[:, win_idx_c]).sum(axis=1).mean(),
    },
    {
        "information": "price_lag_24h (D-1, same hour)",
        "mean_abs_shap": np.abs(
            sv_c.values[:, collapsed_names.index("price_lag_24h")]
        ).mean(),
    },
    {
        "information": "price_lag_48h (D-2, same hour)",
        "mean_abs_shap": np.abs(
            sv_c.values[:, collapsed_names.index("price_lag_48h")]
        ).mean(),
    },
    {
        "information": "price_lag_168h (D-7, same hour)",
        "mean_abs_shap": np.abs(
            sv_c.values[:, collapsed_names.index("price_lag_168h")]
        ).mean(),
    },
]
lag_df = pd.DataFrame(lag_rows)
lag_df["share_of_price_history_pct"] = (
    100 * lag_df["mean_abs_shap"] / lag_df["mean_abs_shap"].sum()
)
display(lag_df)


Left plot (window only)
- Almost no diagonal pattern is visible.
- Reason: the feature `price_lag_24h` carries the exact same "same hour yesterday" value as the window diagonal, so TreeSHAP splits the credit and drains it away from the window.
- The latest window hour (h-1) has by far the highest SHAP values, across almost all target hours, matching with the price-history breakdown plot.
- The second-latest hour (h-2) and the hours around 16:00 of the previous day are the next most important, matching with the price-history breakdown plot.
- No other striking patterns; importance is fairly stable across the different target hours.

Right plot (window + price_lag_24h added onto the diagonal)
- A clear diagonal pattern appears once the twin feature is merged back in.
- This indicates that the model relies on the combined same-hour information from the window and `price_lag_24h`; in the left plot, their correlation distributes the attribution across the two features.
- `price_lag_24h` matters more around midday and clearly less at night.

#### Dependence Plots

In [ ]:
# custom scatter (feature value vs SHAP value), colored by the hour of day
hour = np.asarray(ts_test.hour)
dep_feats = ["load_fc", "price_lag_24h", "price_lag_48h", "price_lag_168h", "U100_fc", "ssrd_fc"]

fig, axes = plt.subplots(3, 2, figsize=(15, 12), sharey=True, constrained_layout=True)  # same y-scale -> magnitudes comparable
for ax, feat in zip(axes.flat, dep_feats):
    j = feature_names.index(feat)
    sc = ax.scatter(X_test[feat], sv.values[:, j], c=hour, cmap="twilight", s=6, alpha=0.6)
    ax.axhline(0, color="grey", lw=0.8)
    ax.set_xlabel(f"{feat} (scaled)")
    ax.set_title(feat)
for ax in axes[:, 0]:                              # ylabel only on the left column (sharey hides inner ticks)
    ax.set_ylabel("SHAP value (asinh scale)")
fig.colorbar(sc, ax=axes, label="hour of day", shrink=0.8)
plt.show()

# hour / weekday dependence: collapsed (net) SHAP vs the ACTUAL value
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, name, xlabel in zip(axes, ["hour", "weekday"], ["hour of day", "weekday (0=Mon, 6=Sun)"]):
    j = collapsed_names.index(name)
    xv, yv = sv_c.data[:, j], sv_c.values[:, j]
    ax.scatter(xv, yv, s=6, alpha=0.25, color="grey")
    means = pd.Series(yv).groupby(xv.astype(int)).mean()
    ax.plot(means.index, means.values, "o-", color="tab:red", label="mean")
    ax.axhline(0, color="black", lw=0.6)
    ax.set_xlabel(xlabel); ax.set_ylabel("SHAP value (asinh scale)")
    ax.set_title(f"{name}: combined SHAP vs actual value"); ax.legend()
plt.tight_layout(); plt.show()


All six attribution patterns are consistent with market logic
- Load and all price lags are associated with positive SHAP values, whereas wind (`U100_fc`) and solar radiation (`ssrd_fc`) are associated with negative SHAP values.
- These model attributions are consistent with expected supply-demand relationships.

Price lags
- `price_lag_24h`: steepest curve, the main anchor.
- `price_lag_168h`: medium slope, clear weekly seasonality.
- `price_lag_48h`: almost flat, just little extra information.
- So: yesterday ≫ last week > two days ago; matches the lag-distance table and the weekly ACF peak from the EDA.
- `price_lag_24h` and `168h` show a step at very low past prices; `48h` does not.

Other variables
- `load_fc`: S-shaped relationship that flattens at high and low load; the SHAP magnitudes are generally larger around midday than at night.
- `U100_fc`: strong negative SHAP values (down to -0.4), with no clear hourly structure.
- `ssrd_fc`: one-sided relationship; the cluster around zero corresponds to nighttime observations without solar radiation. Daytime observations are associated with negative SHAP values, with larger magnitudes around noon than in the evening. Global importance averages over both daytime and nighttime observations.

hour
- Slightly positive at night, clear dip at midday (10-14h), strongest peak in the evening (18-20h).
- The pattern is similar to the hourly plot in the EDA, with a midday dip and an evening peak.
- Spread grows with the size of the effect, but not with specific hour

weekday
- Positive on Mon-Fri, slowly falling; clearly negative on Sat/Sun.
- Weekdays are associated with positive SHAP values and weekends with negative SHAP values, matching the EDA weekday bar plot.

Shapes match the EDA, but are weaker
- The curves have a similar shape to the raw price-by-hour and price-by-weekday patterns from the EDA, indicating that the model uses a comparable calendar structure.
- The SHAP patterns are weaker than the EDA curves, which may be because other variables such as `ssrd_fc` and `load_fc` contain overlapping daily and weekly information.

#### Temporal contribution share

In [ ]:
# Contributions are normalized per prediction (share of the total |SHAP|, in %)
# Shares are scale-free, so no asinh interpretation issue
total_abs = abs_c.sum(axis=1)
share = pd.DataFrame(index=ts_test)
for name, cols in groups.items():
    idx = [collapsed_names.index(c) for c in cols]
    share[name] = abs_c[:, idx].sum(axis=1) / total_abs * 100
share["hour"] = np.asarray(ts_test.hour)

# group share by hour of day: when does which driver matter?
by_hour = share.groupby("hour").mean()
fig, ax = plt.subplots(figsize=(12, 4))
im = ax.imshow(by_hour.T, aspect="auto", cmap="viridis")
ax.set_xticks(range(24)); ax.set_xticklabels(by_hour.index)
ax.set_yticks(range(len(by_hour.columns))); ax.set_yticklabels(by_hour.columns)
ax.set_xlabel("hour of day"); ax.set_title("contribution share (%) per group and hour of day")
fig.colorbar(im, ax=ax, label="share of total |SHAP| (%)")
plt.tight_layout(); plt.show()

# group shares over the test period, with the daily MAE overlaid:
# do error spikes coincide with shifts in what the model relies on?
daily_share = share.drop(columns="hour").resample("D").mean().rolling(7, min_periods=1).mean()
daily_mae = pd.Series(np.abs(y_test - np.asarray(y_pred)), index=ts_test).resample("D").mean().rolling(7, min_periods=1).mean()

fig, ax = plt.subplots(figsize=(13, 4.5))
daily_share.plot(ax=ax)
ax.set_ylabel("contribution share (%)"); ax.set_xlabel("")
ax2 = ax.twinx()
ax2.plot(daily_mae.index, daily_mae.values, color="black", linestyle="--", lw=1.5, alpha=0.8, label="daily MAE")
ax2.set_ylabel("MAE (EUR/MWh)", color="red"); ax2.tick_params(axis="y", labelcolor="red")
ax.set_title("Group contribution shares over the test period (7-day rolling) with daily MAE")
lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax.legend(lines1 + lines2, labels1 + labels2, loc="upper left", fontsize=8)
plt.tight_layout(); plt.show()


intraday trend
- The attribution share of `price lags` rises through the day and peaks in the evening (~19-20h), increasing from about 18% at night to about 37% around the evening price peak. Thus, price lags account for the largest attribution share during these hours.
- `weather forecast` shows a mild opposite trend: a bit higher at night, lower at midday (~10 pp swing).
- `price history window`, `load forecast` and `calendar` keep an almost flat share over all 24 hours.
- So the ordering of drivers is stable all day; only the price-lags / weather balance shifts.

Contribution shares over whole test set
- `price lags` are the top group almost the whole period (~25-35%).
- `price history window` drifts slowly upward toward winter (~18% in summer, ~24% in December).
- `weather forecast` and `price lags` move in opposite directions: when one is high, the other is low.
- `load forecast` and `calendar` stay low and roughly flat all period (~8-17%).

- Visual inspection does not show a clear alignment between MAE peaks and changes in the contribution shares. No stronger conclusion about prediction stability is drawn from this descriptive comparison.

In [ ]:
# Seasonal stability of the variable: summer (Jul-Sep) vs autumn/winter (Oct-Dec)
season_split = pd.Timestamp("2025-10-01", tz=LOCAL_TZ)
seasons = {"Jul-Sep": np.asarray(ts_test < season_split), "Oct-Dec": np.asarray(ts_test >= season_split)}

season_df = pd.DataFrame(index=list(groups))
for label, m in seasons.items():
    tot = abs_c[m].sum(axis=1)
    season_df[label] = [
        (abs_c[np.ix_(m, [collapsed_names.index(c) for c in cols])].sum(axis=1) / tot).mean() * 100
        for cols in groups.values()
    ]

fig, ax = plt.subplots(figsize=(9, 4))
season_df.plot.barh(ax=ax)
ax.set_xlabel("contribution share (%)")
ax.set_title("Group contribution shares by season")
plt.tight_layout(); plt.show()

season_df["diff (Oct-Dec minus Jul-Sep)"] = season_df["Oct-Dec"] - season_df["Jul-Sep"]
display(season_df.round(2))



- The largest seasonal change is ~4 increasing percentage points for price-history
- Weather forecasts, price lags and calenar become slightly less important in the second season
- In Jul-Sep, weather forecasts has biggest contribution, in Oct-Dec it is price history
- In general moderate differences
- Data only half a year

### Local Importance

In [ ]:
# Native shap waterfalls, stacked vertically.The 24 window hours are summed (NET) into one "price history window" bar.

win_idx_c  = [collapsed_names.index(w) for w in window_names]
exo_idx    = [j for j, n in enumerate(collapsed_names) if n not in window_names]
grp_names  = [collapsed_names[j] for j in exo_idx] + [f"price history window ({XGB_LOOKBACK}h)"]
base_const = float(np.ravel(sv_c.base_values)[0])
abs_err    = np.abs(y_test - np.asarray(y_pred))

def sv_local(i):                                           # one instance -> local
    vals = np.append(sv_c.values[i, exo_idx], sv_c.values[i, win_idx_c].sum())
    data = np.append(sv_c.data[i, exo_idx], np.nan)
    return shap.Explanation(values=vals, base_values=base_const, data=data, feature_names=grp_names)

def show_local(i, label):
    pred_eur = float(asinh_inverse(pred_t[i], c_tv))
    print(f"\n=== {label}: {ts_test[i]} | actual={y_test[i]:.2f} EUR/MWh | predicted={pred_eur:.2f} EUR/MWh ===")
    shap.plots.waterfall(sv_local(i), max_display=11)

# --- price extremes + median ---
show_local(int(np.argmax(y_test)), "highest price")
show_local(int(np.argmin(y_test)), "lowest price")
show_local(int(np.argsort(y_test)[len(y_test) // 2]), "median price")

# --- driver extremes (highest / lowest of each top driver) ---
show_local(int(np.argmax(X_test["U100_fc"].to_numpy())),    "highest U100_fc (windiest)")
show_local(int(np.argmin(X_test["U100_fc"].to_numpy())),    "lowest U100_fc (calmest)")
show_local(int(np.argmax(X_test["ssrd_fc"].to_numpy())),    "highest ssrd_fc (max solar)")
show_local(int(np.argmin(X_test["ssrd_fc"].to_numpy())),    "lowest ssrd_fc (no solar)")
show_local(int(np.argmax(X_test["load_fc"].to_numpy())), "highest load_fc")
show_local(int(np.argmin(X_test["load_fc"].to_numpy())), "lowest load_fc")

# --- error extremes ---
show_local(int(np.argmax(abs_err)), "largest error")
show_local(int(np.argmin(abs_err)), "smallest error")

# --- single holiday vs single regular day (both at 12:00, same weekday) ---
hol = X_test["holiday_not_sunday"].to_numpy() > 0.5
wd  = np.asarray(ts_test.dayofweek)
hour_arr = np.asarray(ts_test.hour)
hol_noon = np.flatnonzero(hol & (hour_arr == 12))
if len(hol_noon):
    i_hol = int(hol_noon[0])
    cand = np.flatnonzero((~hol) & (hour_arr == 12) & (wd == wd[i_hol]) & (wd != 6))
    i_reg = int(cand[np.argmin(np.abs(cand - i_hol))])
    show_local(i_hol, "single holiday, 12:00")
    show_local(i_reg, "single regular day (same weekday), 12:00")
else:
    print("No holiday at 12:00 in the test period.")


Price extremes
- Highest price (01.07, 20:00, 476.19 → 204.64): the evening and persistence features have positive SHAP contributions, but the model substantially underestimates the spike. The available features do not produce contributions large enough to reproduce this extreme price.
- Lowest price (10.08, 13:00, -61.08 → -82.34): high solar radiation and low recent prices have negative SHAP contributions, but the prediction is about 21 EUR/MWh below the observed price.
- Median price: a typical hour, contributions are small and balanced.

Driver extremes: each selected driver has a prominent attribution in its corresponding waterfall, with a direction consistent with the global patterns
- `U100_fc`: the highest wind forecast has a strong negative SHAP contribution (observed price near zero), whereas the lowest wind forecast has a positive SHAP contribution.
- `ssrd_fc`: at maximum solar radiation, `ssrd_fc` is the largest negative contribution. The observed price remains moderate (78.14 EUR/MWh), while the model predicts 51.86 EUR/MWh. At night, the `ssrd_fc` contribution is approximately zero and the prediction is primarily attributed to the remaining features.
- `load_fc`: at maximum load, `load_fc` has a positive SHAP contribution, while other features have negative contributions and the model underestimates the observed price (128.04 → 85.02 EUR/MWh). At minimum load, the contributions are relatively small and the prediction is close to the base value.

Error extremes
- Largest error = the price spike itself (same as highest price).
- Smallest error (30.11, 13:00, 82.52 → 82.52): a calm midday hour, predicted almost exactly.

Holiday vs regular (12:00): the holiday indicator has a negative SHAP contribution of approximately -0.07 for the selected holiday observation and a contribution close to zero for the selected regular day. This is a model attribution for these observations, not a causal effect.

Several extremes fall on the first test day (01.07.2025): Highest price (20:00), highest solar (15:00) and lowest solar (03:00), suggesting this day was unusually volatile at the start of the test period.